In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11

In [ ]:
file_path = Path("amazon_prime_titles.csv")
if not file_path.exists():
    from google.colab import files
    uploaded = files.upload()
    file_path = Path(next(iter(uploaded)))

df = pd.read_csv(file_path)
print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
print("Columns:", df.columns.tolist())
display(df.dtypes.astype(str).to_frame("Data type"))
display(df.isna().sum().sort_values(ascending=False).to_frame("Missing values"))
print("Exact duplicate rows:", df.duplicated().sum())

In [ ]:
clean = df.copy()
text_cols = ["director", "cast", "country", "rating", "date_added", "description"]
for col in text_cols:
    clean[col] = clean[col].fillna("Unknown").astype(str).str.strip()

clean["date_added"] = pd.to_datetime(clean["date_added"], errors="coerce")
clean["date_added_year"] = clean["date_added"].dt.year
clean["release_year"] = pd.to_numeric(clean["release_year"], errors="coerce")
clean["duration_value"] = pd.to_numeric(clean["duration"].astype(str).str.extract(r"(\\d+)")[0], errors="coerce")
clean["duration_unit"] = np.where(clean["duration"].astype(str).str.contains("Season", case=False, na=False), "Seasons", "Minutes")
clean["primary_genre"] = clean["listed_in"].astype(str).str.split(",").str[0].str.strip()
clean = clean.drop_duplicates(subset=["show_id"])
print("Rows after removing duplicate show IDs:", len(clean))
display(clean.head())

In [ ]:
type_counts = clean["type"].value_counts()
display(type_counts.to_frame("Title count"))
type_counts.plot(kind="bar", title="Titles by Content Type", ylabel="Number of titles", xlabel="Content type", rot=0)
plt.tight_layout(); plt.show()

In [ ]:
genre_long = clean[["show_id", "listed_in"]].assign(genre=clean["listed_in"].str.split(",")).explode("genre")
genre_long["genre"] = genre_long["genre"].str.strip()
genre_counts = genre_long.groupby("genre")["show_id"].nunique().sort_values(ascending=False)
display(genre_counts.head(15).to_frame("Title count"))
genre_counts.head(10).sort_values().plot(kind="barh", title="Top 10 Genres", xlabel="Number of titles")
plt.tight_layout(); plt.show()

In [ ]:
rating_counts = clean["rating"].replace("Unknown", np.nan).value_counts().head(12)
display(rating_counts.to_frame("Title count"))
rating_counts.sort_values().plot(kind="barh", title="Most Common Content Ratings", xlabel="Number of titles")
plt.tight_layout(); plt.show()

In [ ]:
release_counts = clean.groupby("release_year")["show_id"].nunique().sort_index()
display(release_counts.tail(15).to_frame("Titles released"))
release_counts.tail(30).plot(kind="line", marker="o", title="Release-Year Trend (Last 30 Years in Dataset)", ylabel="Number of titles", xlabel="Release year")
plt.tight_layout(); plt.show()

In [ ]:
country_long = clean[["show_id", "country"]].copy()
country_long["country"] = country_long["country"].replace("Unknown", np.nan).str.split(",")
country_long = country_long.explode("country")
country_long["country"] = country_long["country"].str.strip()
country_counts = country_long.dropna(subset=["country"]).groupby("country")["show_id"].nunique().sort_values(ascending=False)
display(country_counts.head(15).to_frame("Title count"))
country_counts.head(10).sort_values().plot(kind="barh", title="Top 10 Countries by Listed Titles", xlabel="Number of titles")
plt.tight_layout(); plt.show()

In [ ]:
movie_duration = clean.loc[(clean["type"]=="Movie") & clean["duration_unit"].eq("Minutes"), "duration_value"].dropna()
tv_seasons = clean.loc[(clean["type"]=="TV Show") & clean["duration_unit"].eq("Seasons"), "duration_value"].dropna()
print("Movie duration (minutes) – median:", movie_duration.median(), "mean:", round(movie_duration.mean(), 1))
print("TV seasons – median:", tv_seasons.median(), "mean:", round(tv_seasons.mean(), 1))
movie_duration.plot(kind="hist", bins=25, title="Distribution of Movie Runtime", xlabel="Runtime (minutes)")
plt.tight_layout(); plt.show()

In [ ]:
added_counts = clean.dropna(subset=["date_added_year"]).groupby("date_added_year")["show_id"].nunique()
display(added_counts.to_frame("Titles added"))
added_counts.plot(kind="bar", title="Titles by Year Added to Catalogue", xlabel="Year added", ylabel="Number of titles", rot=45)
plt.tight_layout(); plt.show()

In [ ]:
clean.to_csv("amazon_prime_cleaned.csv", index=False)
genre_counts.rename("title_count").to_csv("genre_summary.csv")
country_counts.rename("title_count").to_csv("country_summary.csv")
rating_counts.rename("title_count").to_csv("rating_summary.csv")
print("Exported cleaned dataset and summary CSV files.")